#cpu


In [ ]:
!pip uninstall -y onnxruntime-gpu onnxruntime
!pip install -q onnxruntime opencv-python-headless imageio imageio-ffmpeg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.2 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving avatar_001_mixed_matched.mp4 to avatar_001_mixed_matched.mp4


In [ ]:
import onnxruntime as ort
print(ort.get_available_providers())


['AzureExecutionProvider', 'CPUExecutionProvider']


In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
from time import perf_counter

INPUT_VIDEO = "/content/avatar_001_mixed_matched.mp4"
MODEL_PATH = "/content/GPEN-BFR-256.onnx"
OUTPUT_VIDEO = "/content/gpen256_cpu_output.mp4"
INPUT_SIZE = 256

session = ort.InferenceSession(
    MODEL_PATH,
    providers=["CPUExecutionProvider"],
)
print("providers:", session.get_providers())

input_name = session.get_inputs()[0].name

def preprocess(frame_bgr, input_size=256):
    resized = cv2.resize(frame_bgr, (input_size, input_size))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32)
    rgb = rgb / 255.0
    rgb = rgb * 2.0 - 1.0
    chw = np.transpose(rgb, (2, 0, 1))
    return np.expand_dims(chw, axis=0).astype(np.float32)

def postprocess(output, out_size):
    image = output[0].transpose(1, 2, 0)
    image = ((image + 1.0) * 0.5 * 255.0).clip(0, 255).astype(np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return cv2.resize(image, out_size)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

count = 0
start = perf_counter()

while True:
    ok, frame = cap.read()
    if not ok:
        break

    blob = preprocess(frame, INPUT_SIZE)
    output = session.run(None, {input_name: blob})[0]
    restored = postprocess(output, (width, height))
    writer.write(restored)

    count += 1
    if count % 30 == 0:
        elapsed = perf_counter() - start
        print(f"{count} frames, {count/max(elapsed,1e-6):.2f} fps")

cap.release()
writer.release()

print("saved:", OUTPUT_VIDEO)


providers: ['CPUExecutionProvider']
30 frames, 2.55 fps
60 frames, 2.47 fps
90 frames, 2.50 fps
120 frames, 2.51 fps
150 frames, 2.51 fps
180 frames, 2.52 fps
210 frames, 2.52 fps
240 frames, 2.52 fps
270 frames, 2.53 fps
300 frames, 2.53 fps
330 frames, 2.50 fps
360 frames, 2.51 fps
390 frames, 2.52 fps
420 frames, 2.52 fps
450 frames, 2.53 fps
480 frames, 2.53 fps
510 frames, 2.54 fps
540 frames, 2.54 fps
570 frames, 2.54 fps
saved: /content/gpen256_cpu_output.mp4


In [ ]:
from google.colab import files
files.download("/content/gpen256_cpu_output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>